In [ ]:
%pip install transformers torch scikit-learn pandas tqdm

In [ ]:
%pip install -U "torch>=2.1" "transformers>=4.40" "accelerate>=1.1.0"

In [ ]:

import os
import torch
import pandas as pd
import numpy as np
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    
print(f"🚀 Используемое устройство: {device}")

In [ ]:

def clean_for_bert(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df = pd.read_csv('./Датасет для вкр обучение - output.csv-5.csv', encoding='utf-8')
df['text_clean'] = df['text'].apply(clean_for_bert)
df['label_binary'] = (df['label'] == 'hate_speech').astype(int)

sublabel_to_id = {'ethnic_discrimination': 0, 'religious_intolerance': 1, 'sexism': 2, 'ableism': 3}
df['label_multiclass'] = df['sublabel'].map(sublabel_to_id)

df = df.dropna(subset=['text_clean', 'label_binary']).reset_index(drop=True)
print(f"📊 Датасет загружен: {len(df)} строк")

In [ ]:

df_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label_binary']
)

# Данные для Model 1 (Binary)
X_train_bin = df_train['text_clean'].values
y_train_bin = df_train['label_binary'].values
X_test_bin  = df_test['text_clean'].values
y_test_bin  = df_test['label_binary'].values

df_hate_train = df_train[df_train['label_binary'] == 1].dropna(subset=['label_multiclass'])
X_train_multi = df_hate_train['text_clean'].values
y_train_multi = df_hate_train['label_multiclass'].astype(int).values

df_hate_test = df_test[df_test['label_binary'] == 1].dropna(subset=['label_multiclass'])
X_test_multi = df_hate_test['text_clean'].values
y_test_multi = df_hate_test['label_multiclass'].astype(int).values

In [6]:
# [Ячейка 4] Класс датасета
class HateSpeechDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=126):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [7]:
# [Ячейка 5] Токенизатор и создание датасетов
MODEL_NAME = 'DeepPavlov/rubert-base-cased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset1 = HateSpeechDataset(X_train_bin, y_train_bin, tokenizer)
test_dataset1  = HateSpeechDataset(X_test_bin, y_test_bin, tokenizer)
train_dataset2 = HateSpeechDataset(X_train_multi, y_train_multi, tokenizer)
test_dataset2  = HateSpeechDataset(X_test_multi, y_test_multi, tokenizer)

In [8]:
training_args = TrainingArguments(
    output_dir='./results_hierarchical',
    num_train_epochs=5,                # ⬆️ минимум 3, лучше 4-5
    learning_rate=2e-5,                # ⬇️ BERT лучше учится на 2e-5
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,       # ✅ Автозагрузка лучшей эпохи
    metric_for_best_model='eval_loss', # ✅ Ранний стоп по валидации
    report_to='none',
    disable_tqdm=False,
)

In [ ]:
# [Ячейка 7] Обучение Model 1 (Binary: hate vs non-hate)
model1 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
trainer1 = Trainer(model=model1, args=training_args, train_dataset=train_dataset1, eval_dataset=test_dataset1)

print("🔹 Training Model 1...")
trainer1.train()

In [ ]:
# [Ячейка 8] Обучение Model 2 (Multi-class: 4 подтипа hate)
model2 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4).to(device)
trainer2 = Trainer(model=model2, args=training_args, train_dataset=train_dataset2, eval_dataset=test_dataset2)

print("🔹 Training Model 2...")
trainer2.train()

### Иерархическая оценка

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model1.to(device)
model2.to(device)

In [ ]:
# [Ячейка 9] Корректная иерархическая оценка
def predict_hierarchical(text):
    model1.eval()
    model2.eval()
    
    enc = tokenizer(text, truncation=True, padding='max_length', max_length=128, return_tensors='pt')
    enc = {k: v.to(device) for k, v in enc.items()}
    
    with torch.no_grad():
        logits1 = model1(**enc).logits[0]
        pred1 = torch.argmax(logits1).item()
        
        if pred1 == 1:
            logits2 = model2(**enc).logits[0]
            pred2 = torch.argmax(logits2).item()
            return pred1, pred2
        return pred1, None

correct = 0
total = len(df_test)

for i in range(total):
    text = df_test.iloc[i]['text_clean']
    true_bin = int(df_test.iloc[i]['label_binary'])
    true_multi = df_test.iloc[i]['label_multiclass']  
    
    pred1, pred2 = predict_hierarchical(text)
    
    if true_bin == 0: 
        if pred1 == 0:
            correct += 1
    else:  
        if pred1 == 1 and pred2 == int(true_multi):
            correct += 1

print(f"🎯 ИЕРАРХИЧЕСКАЯ ТОЧНОСТЬ: {correct / total:.4f}")

### Оценка

In [ ]:
%pip install matplotlib

In [ ]:
%pip install seaborn

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

all_preds_bin = []
all_preds_multi = []
all_true_bin = df_test['label_binary'].values
all_true_multi = df_test['label_multiclass'].values

print("Сбор предсказаний RuBERT...")
for i in range(len(df_test)):
    text = df_test.iloc[i]['text_clean']
    p1, p2 = predict_hierarchical(text)
    all_preds_bin.append(p1)
    all_preds_multi.append(p2 if p2 is not None else -1) 

In [ ]:
cm1 = confusion_matrix(all_true_bin, all_preds_bin)
plt.figure(figsize=(8, 6))
sns.heatmap(cm1, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No_hate_speech', 'Hate_speech'], 
            yticklabels=['No_hate_speech', 'Hate_speech'])
plt.title('Матрица ошибок RuBERT (Уровень 1: Бинарный)')
plt.ylabel('Истинные метки')
plt.xlabel('Предсказанные метки')
plt.show()

In [ ]:
print("\n📊 Метрики Уровня 1:")
print(classification_report(all_true_bin, all_preds_bin, target_names=['Non-Hate', 'Hate']))

In [ ]:
hate_indices = np.where(all_true_bin == 1)[0]
true_multi_hate = all_true_multi[hate_indices].astype(int)
preds_multi_hate = np.array(all_preds_multi)[hate_indices]

mask = preds_multi_hate != -1
final_true = true_multi_hate[mask]
final_pred = preds_multi_hate[mask]

target_names = ['Ethnic', 'Religious', 'Sexism', 'Ableism']

cm2 = confusion_matrix(final_true, final_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm2, annot=True, fmt='d', cmap='Greens', 
            xticklabels=target_names, 
            yticklabels=target_names)
plt.title('Матрица ошибок RuBERT (Уровень 2: Категории хейта)')
plt.ylabel('Истинные метки')
plt.xlabel('Предсказанные метки')
plt.show()

In [ ]:
print("\n📊 Метрики Уровня 2:")
print(classification_report(final_true, final_pred, target_names=target_names))

### SHAP визуализация

In [ ]:
%pip install lime

In [ ]:
import torch
import numpy as np
from lime.lime_text import LimeTextExplainer

def predictor(texts):
    model2.eval()
    device = torch.device("cpu") 
    model2.to(device)
    
    all_probs = []
    batch_size = 10 
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, 
                          truncation=True, max_length=128).to(device)
        
        with torch.no_grad():
            outputs = model2(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            all_probs.append(probs.numpy())
            
    return np.vstack(all_probs)

class_names = ['ethnic', 'religious', 'sexism', 'ableism'] 
explainer = LimeTextExplainer(class_names=class_names)

target_id = 663
example_text = df[df['id'] == target_id]['text'].values[0]

print(f"Анализ текста: {example_text}")

exp = explainer.explain_instance(
    example_text, 
    predictor, 
    num_features=6, 
    labels=[2],      
    num_samples=150 
)

from IPython.display import HTML

raw_html = exp.as_html()

styled_html = f"""
<div style="background-color: white; color: black; padding: 20px; border-radius: 10px;">
    {raw_html}
</div>
"""

display(HTML(styled_html))

In [ ]:
import pandas as pd
import numpy as np

results_df = df_test.copy()
results_df['pred_bin'] = all_preds_bin
results_df['pred_multi'] = all_preds_multi

correct_mask = (
    (results_df['label_binary'] == 1) & 
    (results_df['pred_bin'] == 1) & 
    (results_df['label_multiclass'] == results_df['pred_multi'])
)

lime_candidates = results_df[correct_mask].sort_index()

sublabel_map = {0: 'ethnic', 1: 'religious', 2: 'sexism', 3: 'ableism'}
lime_candidates['category_name'] = lime_candidates['label_multiclass'].map(sublabel_map)

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

print(f"Найдено верно классифицированных иерархических примеров: {len(lime_candidates)}")


if len(lime_candidates) > 0:

    display(lime_candidates[['text', 'category_name']])
else:
    print("Примеров не найдено. Проверь соответствие меток в label_multiclass и pred_multi.")

lime_candidates[['text', 'category_name']].to_csv('candidates_for_lime.csv', index=True, encoding='utf-8-sig')

print("Файл 'candidates_for_lime.csv' успешно создан!")

In [ ]:

first_level_correct = results_df[results_df['label_binary'] == results_df['pred_bin']].copy()

true_negatives = first_level_correct[first_level_correct['label_binary'] == 0]
true_positives = first_level_correct[first_level_correct['label_binary'] == 1]

print(f"Всего верных предсказаний на 1-м уровне: {len(first_level_correct)}")
print(f"Из них верно определено как 'Нейтрально' (TN): {len(true_negatives)}")
print(f"Из них верно определено как 'Токсично' (TP): {len(true_positives)}")

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

print("\n--- ПРИМЕРЫ ВЕРНО ОПРЕДЕЛЕННОГО НЕЙТРАЛЬНОГО КОНТЕКСТА (Первый уровень) ---")
if len(true_negatives) > 0:

    display(true_negatives[['text']].sample(min(15, len(true_negatives)), random_state=42))


true_negatives[['text']].to_csv('all_correct_neutral.csv', index=True, encoding='utf-8-sig')

### Бейзлайн

In [96]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import joblib

In [97]:
tfidf = joblib.load('tfidf_baseline.pkl')
model_l1 = joblib.load('model_l1_baseline.pkl')
model_l2 = joblib.load('model_l2_baseline.pkl')

### Сравнение моделей (примеры)

In [ ]:
%pip install emoji

In [ ]:
%pip install nltk pymorphy3

In [46]:
model1.to('cpu')
device = 'cpu'

In [ ]:
import torch
import numpy as np
from lime.lime_text import LimeTextExplainer
from IPython.display import HTML

def predictor(texts):
    model1.eval() 
    device = torch.device("cpu") 
    model1.to(device)
    
    all_probs = []
    batch_size = 12 
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, 
                          truncation=True, max_length=128).to(device)
        
        with torch.no_grad():
            outputs = model1(**inputs)
           
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            all_probs.append(probs.cpu().numpy())
            
    return np.vstack(all_probs)

class_names = ['No hate speech', 'Hate speech'] 
explainer = LimeTextExplainer(class_names=class_names)

target_id = 3030

example_text = results_df.loc[target_id, 'text']

print(f"Анализ текста: {example_text}")

exp = explainer.explain_instance(
    example_text, 
    predictor, 
    num_features=8,   
    labels=[1],       
    num_samples=800   
)

raw_html = exp.as_html()
styled_html = f"""
<div style="background-color: white; color: black; padding: 15px; border: 1px solid #ddd;">
    {raw_html}
</div>
"""

display(HTML(styled_html))


In [ ]:
import pandas as pd
import numpy as np

results_df = df_test.copy()
results_df['pred_bin'] = all_preds_bin
results_df['pred_multi'] = all_preds_multi

fp_mask = (results_df['label_binary'] == 0) & (results_df['pred_bin'] == 1)

fn_mask = (results_df['label_binary'] == 1) & (results_df['pred_bin'] == 0)

wrong_cat_mask = (
    (results_df['label_binary'] == 1) & 
    (results_df['pred_bin'] == 1) & 
    (results_df['label_multiclass'] != results_df['pred_multi'])
)

errors_df = results_df[fp_mask | fn_mask | wrong_cat_mask].copy()

def identify_error(row):
    if row['label_binary'] == 0 and row['pred_bin'] == 1:
        return 'False Positive (Ghost Hate)'
    if row['label_binary'] == 1 and row['pred_bin'] == 0:
        return 'False Negative (Missed Hate)'
    if row['label_multiclass'] != row['pred_multi']:
        return 'Wrong Category'
    return 'Correct'

errors_df['error_type'] = errors_df.apply(identify_error, axis=1)

sublabel_map = {0: 'ethnic', 1: 'religious', 2: 'sexism', 3: 'ableism'}
errors_df['true_category_name'] = errors_df['label_multiclass'].map(sublabel_map)
errors_df['pred_category_name'] = errors_df['pred_multi'].map(sublabel_map)

pd.set_option('display.max_colwidth', 100)
print(f"Всего найдено ошибок модели: {len(errors_df)}")

if len(errors_df) > 0:

    display(errors_df[['text', 'error_type', 'true_category_name', 'pred_category_name']].sort_values('error_type'))
    
    errors_df.to_csv('bert_errors_analysis.csv', index=True, encoding='utf-8-sig')
    print("\nФайл 'bert_errors_analysis.csv' успешно создан!")
else:
    print("Ошибок не найдено")

In [99]:
import nltk

# Скачиваем необходимые пакеты данных
nltk.download('punkt')
nltk.download('punkt_tab') # В новых версиях NLTK может потребоваться этот пакет
nltk.download('stopwords')

print("Ресурсы NLTK успешно загружены!")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/sofyapetrova/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/sofyapetrova/nltk_data...


Ресурсы NLTK успешно загружены!


[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/sofyapetrova/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
import pandas as pd
import numpy as np
import re
import emoji
import nltk
import pymorphy3
from nltk.corpus import stopwords

nltk.download('punkt')
nltk.download('stopwords')

morph = pymorphy3.MorphAnalyzer()
russian_stopwords = set(stopwords.words('russian'))


def clean_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = emoji.replace_emoji(text, replace='')
    text = re.sub(r'[^а-яА-Яa-zA-Z0-9\s\.\,\!\?\;\:\-\"\'\(\)\*]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def preprocess_for_baseline(text):
    """Полный цикл для классического ML"""
    cleaned = clean_text(text)
    tokens = nltk.word_tokenize(cleaned, language='russian')
    lemmas = []
    for token in tokens:
        if len(token) < 2: continue 
        lemma = morph.parse(token)[0].normal_form
        if lemma not in russian_stopwords:
            lemmas.append(lemma)
    return ' '.join(lemmas)

res = df_test.copy()

print("Запуск предобработки для Baseline...")
res['text_for_base'] = res['text'].apply(preprocess_for_baseline)

X_tfidf_final = tfidf.transform(res['text_for_base'])
b_bin = model_l1.predict(X_tfidf_final)
b_multi = model_l2.predict(X_tfidf_final)

def clean_val(v):
    if v is None or pd.isna(v): return "-1"
    return str(v).split('.')[0].strip()

sub_to_id = {'ethnic': '0', 'religious': '1', 'sexism': '2', 'ableism': '3'}

res['true_b'] = res['label_binary'].apply(clean_val)
res['true_m'] = res['label_multiclass'].apply(clean_val)

if isinstance(res['sublabel'].iloc[0], str):
    res['true_m'] = res['sublabel'].map(sub_to_id).apply(clean_val)

res['base_b'] = pd.Series(b_bin, index=res.index).apply(clean_val)
res['base_m'] = pd.Series(b_multi, index=res.index).apply(clean_val)

if isinstance(b_multi[0], str):
    res['base_m'] = res['base_m'].map(sub_to_id).fillna(res['base_m'])

res['bert_b'] = pd.Series(bert_bin_list, index=res.index).apply(clean_val)
res['bert_m'] = pd.Series(bert_multi_list, index=res.index).apply(clean_val)

res['BASE_OK'] = (res['base_b'] == res['true_b']) & (res['base_m'] == res['true_m'])
res['BERT_OK'] = (res['bert_b'] == res['true_b']) & (res['bert_m'] == res['true_m'])

win_mask = (res['true_b'] == "1") & (res['BERT_OK'] == True) & (res['BASE_OK'] == False)
bert_wins = res[win_mask].sort_index().copy()

id_to_sub = { '0': 'ethnic', '1': 'religious', '2': 'sexism', '3': 'ableism', '-1': 'none/missed' }
bert_wins['cat_true'] = bert_wins['true_m'].map(id_to_sub)
bert_wins['cat_base'] = bert_wins['base_m'].map(id_to_sub)
bert_wins['cat_bert'] = bert_wins['bert_m'].map(id_to_sub)

bert_wins.loc[bert_wins['base_b'] == "0", 'cat_base'] += ' (missed bin)'

print(f"BERT после полной предобработки: {len(bert_wins)}")
pd.set_option('display.max_colwidth', None)
display(bert_wins[['text', 'cat_true', 'cat_base', 'cat_bert']])